# Reporte de Rendimiento y Benchmarking
## Comparativa: Ejecución Secuencial vs. Grafo Concurrente & Procesamiento Batch de Task Handlers

In [ ]:
import asyncio
import os
import sys
import time
from statistics import mean, median

import matplotlib.pyplot as plt
import nest_asyncio
import pandas as pd

# Permitir asyncio en Jupyter Notebooks
nest_asyncio.apply()

# Configurar sys.path para importar app
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from app.backend.schemas.chat import ChatRequest
from app.backend.services.dispute_service import dispute_service
from app.backend.services.task_handlers import handle_consolidate_financial_summary

# Configuración estática de Matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

### 1. Funciones Principales de Medición

In [ ]:
async def measure_single_request(message: str, use_graph: bool) -> float:
    request = ChatRequest(message=message, session_id="BENCHMARK_SESS_SINGLE")
    start = time.perf_counter()
    await dispute_service.process_chat_message(request, use_graph=use_graph)
    return (time.perf_counter() - start) * 1000.0

async def run_concurrent_users(num_users: int, message: str, use_graph: bool) -> float:
    tasks = [
        dispute_service.process_chat_message(
            ChatRequest(message=message, session_id=f"BENCHMARK_SESS_{i}"),
            use_graph=use_graph,
        )
        for i in range(num_users)
    ]
    start_total = time.perf_counter()
    await asyncio.gather(*tasks)
    return (time.perf_counter() - start_total) * 1000.0

async def measure_task_handler_batch_vs_sequential(num_items: int = 50):
    request = ChatRequest(message="Consolidar resumen financiero", session_id="BENCHMARK_S3_BATCH")
    mock_products = [{"product_id": f"P_{i}", "current_balance": 1500.50 + i} for i in range(num_items)]

    # Secuencial
    start_seq = time.perf_counter()
    for item in mock_products:
        single_context = {
            "products": [item],
            "exchange_rate_usd": 1.0,
            "recent_transactions": [],
            "active_complaints": [],
        }
        await handle_consolidate_financial_summary(context=single_context, request=request, entities={})
    seq_time = (time.perf_counter() - start_seq) * 1000.0

    # Batch
    batch_context = {
        "products": mock_products,
        "exchange_rate_usd": 1.0,
        "recent_transactions": [],
        "active_complaints": [],
    }
    start_batch = time.perf_counter()
    await handle_consolidate_financial_summary(context=batch_context, request=request, entities={})
    batch_time = (time.perf_counter() - start_batch) * 1000.0

    improvement = ((seq_time - batch_time) / seq_time) * 100.0 if seq_time > 0 else 0.0
    return seq_time, batch_time, improvement

### 2. Ejecución del Benchmark y Recolección de Datos

In [ ]:
message = "Quiero ver mi resumen de cuenta"
single_iterations = 10
concurrent_rounds = 5
concurrency_levels = [5, 10, 25, 50]

# Warmup
await measure_single_request(message, use_graph=False)
await measure_single_request(message, use_graph=True)

# 1. Datos Latencia Individual
seq_times = []
graph_times = []
for _ in range(single_iterations):
    seq_times.append(await measure_single_request(message, use_graph=False))
    graph_times.append(await measure_single_request(message, use_graph=True))

# 2. Datos Carga Concurrente
concurrent_results = []
for concurrency in concurrency_levels:
    seq_rounds = []
    graph_rounds = []
    for _ in range(concurrent_rounds):
        seq_rounds.append(await run_concurrent_users(concurrency, message, use_graph=False))
        graph_rounds.append(await run_concurrent_users(concurrency, message, use_graph=True))
    
    concurrent_results.append({
        "concurrency": concurrency,
        "seq_avg": mean(seq_rounds),
        "seq_med": median(seq_rounds),
        "graph_avg": mean(graph_rounds),
        "graph_med": median(graph_rounds),
    })

# 3. Datos Task Handler (Batch vs Secuencial)
seq_th_time, batch_th_time, th_improvement = await measure_task_handler_batch_vs_sequential(num_items=50)
print("✅ Benchmark ejecutado correctamente y datos almacenados.")

### 3. Visualizaciones con Matplotlib

#### Gráfico 1: Comparativa de Latencia Individual (1 Usuario)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

iterations = list(range(1, single_iterations + 1))
ax.plot(iterations, seq_times, marker='o', linewidth=2, label='Ejecución Secuencial', color='#e74c3c')
ax.plot(iterations, graph_times, marker='s', linewidth=2, label='Grafo Concurrente', color='#2ecc71')

ax.set_title('Latencia por Iteración (1 Usuario)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Número de Iteración', fontsize=11)
ax.set_ylabel('Tiempo de Respuesta (ms)', fontsize=11)
ax.legend(frameon=True, facecolor='white', framealpha=0.9)
ax.set_xticks(iterations)


plt.tight_layout()
plt.savefig('grafico1_latencia_individual.png', dpi=300, bbox_inches='tight')
plt.show()

#### Gráfico 2: Tiempos de Respuesta bajo Carga Concurrente

In [ ]:
df_conc = pd.DataFrame(concurrent_results)

fig, ax = plt.subplots(figsize=(9, 5))

bar_width = 0.35
x = range(len(df_conc))

bars1 = ax.bar([i - bar_width/2 for i in x], df_conc['seq_avg'], bar_width, label='Secuencial (Media)', color='#34495e')
bars2 = ax.bar([i + bar_width/2 for i in x], df_conc['graph_avg'], bar_width, label='Grafo Concurrente (Media)', color='#3498db')

# Etiquetas con valores sobre las barras
for bar in bars1:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 5, f'{yval:.0f} ms', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 5, f'{yval:.0f} ms', ha='center', va='bottom', fontsize=9)

ax.set_title('Escalabilidad y Carga Concurrente', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Usuarios Concurrentes', fontsize=11)
ax.set_ylabel('Tiempo Total de Respuesta (ms)', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(df_conc['concurrency'])
ax.legend(frameon=True, facecolor='white', framealpha=0.9)


plt.tight_layout()
plt.savefig('grafico2_carga_concurrente.png', dpi=300, bbox_inches='tight')
plt.show()

#### Gráfico 3: Task Handler - Procesamiento Secuencial vs. Batch

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

categories = ['Secuencial (50 llamadas)', 'Batch (1 llamada)']
times = [seq_th_time, batch_th_time]
colors = ['#e67e22', '#1abc9c']

bars = ax.bar(categories, times, color=colors, width=0.5)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + (yval * 0.02), f'{yval:.2f} ms', ha='center', va='bottom', fontweight='bold')

ax.set_title(f'Eficiencia del Task Handler (Mejora: {th_improvement:.1f}%)', fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Tiempo de Procesamiento (ms)', fontsize=11)

# ... código previo de la gráfica 3 ...

plt.tight_layout()
plt.savefig('grafico3_task_handler_batch.png', dpi=300, bbox_inches='tight')
plt.show()